# WikiArt Style Classifier (Improved Version)

This is a **new notebook** created to test improvements over the previous version.

Why a new notebook?
- The earlier model reached around 50-60% validation accuracy
- We want better learning and better generalization

In this version, we keep the code beginner-friendly and add practical improvements:
- stronger but simple data augmentation
- clean train/validation/test workflow
- pretrained ResNet fine-tuning

## 1. Project Overview

Goal: predict the artistic style of a painting (for example Impressionism, Cubism, Realism) from an image using PyTorch and torchvision.

We will:
- automatically find WikiArt CSV files and image folders
- build a custom `Dataset`
- train a pretrained CNN (ResNet18 or ResNet50)
- track train and validation metrics every epoch
- evaluate on an optional test split made from validation

## 2. Import Libraries

These imports cover data loading, image transforms, modeling, and training.

In [1]:
from pathlib import Path
import random
import warnings
import copy
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Ignore PIL warning for very large source images (we resize before training).
warnings.simplefilter("ignore", Image.DecompressionBombWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 3. Dataset Explanation and Preview

WikiArt data is organized in two ways:
- image folders by style (for example `Impressionism/...jpg`)
- CSV files with `relative_path,label` (no header)

The code below automatically finds the project root and the train/validation CSV files.

In [2]:
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "datasets").exists() and (candidate / "README.md").exists():
            return candidate
    return start.resolve()

def find_split_csv(data_dir: Path, split: str) -> Path:
    split = split.lower()
    csvs = sorted(data_dir.glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    preferred = [p for p in csvs if split in p.stem.lower() and "style" in p.stem.lower()]
    if preferred:
        return preferred[0]

    fallback = [p for p in csvs if split in p.stem.lower()]
    if fallback:
        return fallback[0]

    raise FileNotFoundError(f"Could not find a '{split}' CSV in {data_dir}")

project_root = find_project_root(Path.cwd())
wikiart_dir = project_root / "datasets" / "Wikiart"

train_csv = find_split_csv(wikiart_dir, "train")
val_csv = find_split_csv(wikiart_dir, "val")

train_df = pd.read_csv(train_csv, header=None, names=["relative_path", "label"])
val_df = pd.read_csv(val_csv, header=None, names=["relative_path", "label"])

print(f"Project root: {project_root}")
print(f"WikiArt dir:  {wikiart_dir}")
print(f"Train CSV:    {train_csv.name}")
print(f"Val CSV:      {val_csv.name}")
print(f"Train rows:   {len(train_df):,}")
print(f"Val rows:     {len(val_df):,}")

display(train_df.head())

Project root: C:\Users\Thijs\Desktop\Ai Art Critic
WikiArt dir:  C:\Users\Thijs\Desktop\Ai Art Critic\datasets\Wikiart
Train CSV:    style_train.csv
Val CSV:      style_val.csv
Train rows:   57,025
Val rows:     24,421


,relative_path,label
0,Impressionism/edgar-degas_landscape-on-the-orn...,12
1,Realism/camille-corot_mantes-cathedral.jpg,21
2,Abstract_Expressionism/gene-davis_untitled-197...,0
3,Symbolism/kuzma-petrov-vodkin_in-the-1920.jpg,24
4,Impressionism/maurice-prendergast_paris-boulev...,12


In [3]:
# Build a label -> style name map using the folder name in relative path
def extract_style_name(relative_path: str) -> str:
    return Path(relative_path).parts[0]

train_df["style_name"] = train_df["relative_path"].map(extract_style_name)
val_df["style_name"] = val_df["relative_path"].map(extract_style_name)

label_to_style = (
    train_df[["label", "style_name"]]
    .drop_duplicates()
    .sort_values("label")
    .set_index("label")["style_name"]
    .to_dict()
)

num_classes = len(label_to_style)
print(f"Number of style classes: {num_classes}")
print("First 10 class mappings:")
for lbl, style in list(label_to_style.items())[:10]:
    print(f"{lbl:2d} -> {style}")

Number of style classes: 27
First 10 class mappings:
 0 -> Abstract_Expressionism
 1 -> Action_painting
 2 -> Analytical_Cubism
 3 -> Art_Nouveau_Modern
 4 -> Baroque
 5 -> Color_Field_Painting
 6 -> Contemporary_Realism
 7 -> Cubism
 8 -> Early_Renaissance
 9 -> Expressionism


## 4. Image Transformations (Augmentation + Normalization)

For training, we add simple augmentation to improve generalization:
- random horizontal flip
- small random rotation
- light color jitter

For validation/test, we use only deterministic transforms.

All splits are normalized with ImageNet mean/std because we fine-tune a pretrained model.

In [4]:
image_size = 224
batch_size = 32
num_workers = 0  # safer default for Windows

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

## 5. Custom Dataset Class

This dataset reads image paths from CSV rows and returns `(image_tensor, label)`.

It also handles missing/corrupt images by retrying another sample.

In [5]:
class WikiArtStyleDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, image_root: Path, transform=None, max_retries: int = 10):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_root = Path(image_root)
        self.transform = transform
        self.max_retries = max_retries

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        last_error = None

        for _ in range(self.max_retries):
            row = self.df.iloc[idx]
            img_path = self.image_root / row["relative_path"]
            label = int(row["label"])

            try:
                image = Image.open(img_path).convert("RGB")
                if self.transform:
                    image = self.transform(image)
                return image, label
            except (FileNotFoundError, UnidentifiedImageError, OSError, ValueError) as exc:
                last_error = exc
                idx = random.randint(0, len(self.df) - 1)

        raise RuntimeError(f"Could not load image after {self.max_retries} retries. Last error: {last_error}")

## 6. Create DataLoaders (Train / Validation / Optional Test)

The dataset has train + validation CSVs only.

If enabled, we split **15% of validation** into a small test set for final evaluation.

In [6]:
def filter_existing_rows(df: pd.DataFrame, image_root: Path, split_name: str) -> pd.DataFrame:
    full_paths = df["relative_path"].map(lambda p: image_root / p)
    exists_mask = full_paths.map(lambda p: p.exists())

    removed = int((~exists_mask).sum())
    kept = int(exists_mask.sum())
    print(f"{split_name}: kept {kept:,}, removed {removed:,} missing files")

    cleaned = df.loc[exists_mask].reset_index(drop=True)
    if cleaned.empty:
        raise RuntimeError(f"No valid rows left for split: {split_name}")
    return cleaned

train_df_clean = filter_existing_rows(train_df, wikiart_dir, "train")
val_df_clean = filter_existing_rows(val_df, wikiart_dir, "validation")

CREATE_TEST_SPLIT = True
TEST_FRACTION_FROM_VAL = 0.15  # 10-20% is a good range

if CREATE_TEST_SPLIT and len(val_df_clean) > 20:
    test_df = val_df_clean.sample(frac=TEST_FRACTION_FROM_VAL, random_state=SEED)
    val_eval_df = val_df_clean.drop(test_df.index).reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
else:
    val_eval_df = val_df_clean
    test_df = None

train_dataset = WikiArtStyleDataset(train_df_clean, wikiart_dir, transform=train_transform)
val_dataset = WikiArtStyleDataset(val_eval_df, wikiart_dir, transform=eval_transform)
test_dataset = (
    WikiArtStyleDataset(test_df, wikiart_dir, transform=eval_transform)
    if test_df is not None else None
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = (
    DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    if test_dataset is not None else None
)

print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples:   {len(val_dataset):,}")
if test_dataset is not None:
    print(f"Test samples:  {len(test_dataset):,} (from validation split)")
else:
    print("Test set: not created")

train: kept 57,023, removed 2 missing files
validation: kept 24,421, removed 0 missing files
Train samples: 57,023
Val samples:   20,758
Test samples:  3,663 (from validation split)


## 7. Load Pretrained CNN and Fine-Tune

Pick a model (`resnet18` or `resnet50`), load pretrained weights, and replace the final layer with `num_classes` outputs.

In [7]:
MODEL_NAME = "resnet18"  # change to "resnet50" if you want a larger model
LEARNING_RATE = 3e-4
EPOCHS = 4  # 3-5 is a good quick testing range

def build_model(model_name: str, num_classes: int):
    model_name = model_name.lower()

    if model_name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
    elif model_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)
    else:
        raise ValueError("MODEL_NAME must be 'resnet18' or 'resnet50'")

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = build_model(MODEL_NAME, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Model: {MODEL_NAME}")
print(model.fc)

Model: resnet18
Linear(in_features=512, out_features=27, bias=True)


## 8. Training Loop (Print Train + Validation Metrics)

Each epoch prints:
- train loss, train accuracy
- validation loss, validation accuracy

We also keep the best model checkpoint in memory (by validation accuracy).

In [8]:
def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(is_train):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc

history = []
best_val_acc = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_one_epoch(model, train_loader, criterion, optimizer=optimizer)
    val_loss, val_acc = run_one_epoch(model, val_loader, criterion, optimizer=None)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}"
    )

history_df = pd.DataFrame(history)
display(history_df)

Epoch 01/4 | train_loss=1.7657, train_acc=0.422 | val_loss=1.5473, val_acc=0.483
Epoch 02/4 | train_loss=1.4979, train_acc=0.498 | val_loss=1.4966, val_acc=0.496
Epoch 03/4 | train_loss=1.3626, train_acc=0.541 | val_loss=1.4396, val_acc=0.524
Epoch 04/4 | train_loss=1.2540, train_acc=0.574 | val_loss=1.4387, val_acc=0.530


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,1.765655,0.421795,1.547329,0.483043
1,2,1.497857,0.498150,1.496601,0.495857
2,3,1.362573,0.540834,1.439580,0.523654
3,4,1.253956,0.573804,1.438677,0.529964


## 9. Validation and Optional Test Evaluation

Now we evaluate final model quality:
- best validation accuracy
- optional test accuracy (if we created a test split from validation)

In [9]:
if best_state is not None:
    model.load_state_dict(best_state)

def evaluate_accuracy(model, loader):
    model.eval()
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(dim=1)

            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    return total_correct / total_samples

val_acc_final = evaluate_accuracy(model, val_loader)
print(f"Best validation accuracy: {val_acc_final:.3f}")

if test_loader is not None:
    test_acc_final = evaluate_accuracy(model, test_loader)
    print(f"Test accuracy (from val split): {test_acc_final:.3f}")

Best validation accuracy: 0.530
Test accuracy (from val split): 0.530


## 10. Predict a Single Image

This helper function runs inference on one image and returns the predicted style label and name.

In [10]:
def predict_image_style(model, image_path: Path, transform, label_to_style_map):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor)
        pred_label = int(outputs.argmax(dim=1).item())

    pred_style = label_to_style_map.get(pred_label, f"Unknown label {pred_label}")
    return pred_label, pred_style

if test_df is not None and len(test_df) > 0:
    sample_row = test_df.iloc[0]
else:
    sample_row = val_eval_df.iloc[0]

sample_relative_path = sample_row["relative_path"]
true_label = int(sample_row["label"])
sample_image_path = wikiart_dir / sample_relative_path

pred_label, pred_style = predict_image_style(model, sample_image_path, eval_transform, label_to_style)
true_style = label_to_style.get(true_label, f"Unknown label {true_label}")

print(f"Image:           {sample_relative_path}")
print(f"True style:      {true_style} ({true_label})")
print(f"Predicted style: {pred_style} ({pred_label})")

Image:           Romanticism/gustave-dore_don-quixote-20.jpg
True style:      Romanticism (23)
Predicted style: Romanticism (23)


## 11. Next Steps (Web App Integration)

You can integrate this model into your web app like this:
1. Save the trained model weights
2. Create a backend API endpoint for image upload
3. Preprocess uploaded image with the same `eval_transform`
4. Run model inference and return predicted style
5. Show result in the frontend with confidence score

After this testing notebook, a good next upgrade is adding per-class metrics and confusion matrix to understand which styles are hardest to classify.